# Notebook 02: Bronze full snapshots to governed Silver current state

This notebook implements the SAP Finance Silver boundary:

- Reads only the pipeline run-date full snapshot for each source object
- Standardises and validates source values against explicit contracts
- Upserts valid rows into one current Silver row per SAP business key
- Appends invalid rows to quarantine and operational evidence to audit
- Supports same-day replay through idempotent business-key MERGE

Silver does not infer source deletes because the selected SAP projections do not carry reliable deletion semantics.


In [ ]:
processing_date = ""


In [ ]:
from pyspark.sql import functions as F, types as T, Window
from functools import reduce
from datetime import datetime, timezone
import uuid

BRZ_SLV_LH = "lh_brz_slv_finance"
BRONZE_ROOT = "Files/bronze/sap_finance"
SLV = f"{BRZ_SLV_LH}.slv"
QTN = f"{BRZ_SLV_LH}.quarantine"
AUD = f"{BRZ_SLV_LH}.audit"
SOURCE_SYSTEM = "SAP_S4HANA"
BATCH_RUN_ID = str(uuid.uuid4())
RUN_STARTED_UTC = datetime.now(timezone.utc).replace(tzinfo=None)

processing_date = processing_date.strip()
if not processing_date:
    processing_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")
try:
    PROCESSING_DATE = datetime.strptime(processing_date, "%Y-%m-%d").date()
except ValueError as exc:
    raise ValueError("processing_date must use yyyy-MM-dd format") from exc
PROCESSING_DATE_KEY = PROCESSING_DATE.strftime("%Y%m%d")

# Operational controls
SKIP_SUCCESSFULLY_PROCESSED_FILES = False
FAIL_NOTEBOOK_ON_TABLE_ERROR = True
ALLOW_ADDITIVE_SOURCE_COLUMNS = False

spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.autoCompact.enabled", "true")

for namespace in ["slv", "quarantine", "audit"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {namespace}")

print(f"batch_run_id={BATCH_RUN_ID}")
print(f"run_started_utc={RUN_STARTED_UTC}")
print(f"processing_date={PROCESSING_DATE.isoformat()}")

## Table contracts

The contract defines the allowed source projection, required fields, business key and target datatype for each SAP object. Key-like SAP values remain strings so leading zeros are preserved.

In [ ]:
TABLE_CONFIG = {
    "acdoca": {
        "columns": ["RCLNT","RLDNR","RBUKRS","GJAHR","POPER","BELNR","DOCLN","RACCT","RCNTR","PRCTR","KOKRS","LIFNR","KUNNR","BUDAT","BLDAT","DRCRK","BLART","AWTYP","SGTXT","RHCUR","RKCUR","RWCUR","HSL","KSL","TSL","WSL","MSL"],
        "required": ["RBUKRS","GJAHR","BELNR","DOCLN","RACCT"],
        "keys": ["RCLNT","RLDNR","RBUKRS","GJAHR","BELNR","DOCLN"],
        "types": {"GJAHR":"int","POPER":"int","BUDAT":"date","BLDAT":"date","HSL":"decimal(23,2)","KSL":"decimal(23,2)","TSL":"decimal(23,2)","WSL":"decimal(23,2)","MSL":"decimal(23,3)"},
        "period_columns": ["POPER"], "allowed": {"DRCRK":["S","H"]}
    },
    "bkpf": {
        "columns": ["MANDT","BUKRS","GJAHR","BELNR","MONAT","BUDAT","BLDAT","BLART","STBLG","BKTXT"],
        "required": ["BUKRS","GJAHR","BELNR"],
        "keys": ["MANDT","BUKRS","GJAHR","BELNR"],
        "types": {"GJAHR":"int","MONAT":"int","BUDAT":"date","BLDAT":"date"},
        "period_columns": ["MONAT"], "allowed": {}
    },
    "t001": {
        "columns": ["MANDT","BUKRS","BUTXT","ORT01","LAND1","WAERS","KTOPL"],
        "required": ["BUKRS"], "keys": ["MANDT","BUKRS"], "types": {}, "period_columns": [], "allowed": {}
    },
    "csks": {
        "columns": ["MANDT","KOKRS","KOSTL","BUKRS","PRCTR","VERAK","KOSAR","KHINR","DATAB","DATBI"],
        "required": ["KOKRS","KOSTL"], "keys": ["MANDT","KOKRS","KOSTL"],
        "types": {"DATAB":"date","DATBI":"date"}, "period_columns": [], "allowed": {}
    },
    "cepc": {
        "columns": ["MANDT","KOKRS","PRCTR","BUKRS","VERAK","KHINR","SEGMENT","DATAB","DATBI"],
        "required": ["KOKRS","PRCTR","KHINR"], "keys": ["MANDT","KOKRS","PRCTR"],
        "types": {"DATAB":"date","DATBI":"date"}, "period_columns": [], "allowed": {}
    },
    "ska1": {
        "columns": ["MANDT","KTOPL","SAKNR","XBILK","GVTYP","KTOKS"],
        "required": ["KTOPL","SAKNR"], "keys": ["MANDT","KTOPL","SAKNR"], "types": {}, "period_columns": [], "allowed": {}
    },
    "skat": {
        "columns": ["MANDT","SPRAS","KTOPL","SAKNR","TXT20","TXT50"],
        "required": ["SPRAS","KTOPL","SAKNR"], "keys": ["MANDT","SPRAS","KTOPL","SAKNR"], "types": {}, "period_columns": [], "allowed": {}
    },
    "lfa1": {
        "columns": ["MANDT","LIFNR","NAME1","NAME2","LAND1","ORT01","REGIO","KTOKK","STCD1"],
        "required": ["LIFNR"], "keys": ["MANDT","LIFNR"], "types": {}, "period_columns": [], "allowed": {}
    },
    "kna1": {
        "columns": ["MANDT","KUNNR","NAME1","NAME2","LAND1","ORT01","REGIO","KTOKD","STCD1"],
        "required": ["KUNNR"], "keys": ["MANDT","KUNNR"], "types": {}, "period_columns": [], "allowed": {}
    },
    "faglflexp": {
        "columns": ["RCLNT","RLDNR","RBUKRS","RYEAR","RACCT","PRCTR","KOKRS","VERSN","CURTYPE","RHCUR"] + [f"TSL{i:02d}" for i in range(1,17)],
        "required": ["RBUKRS","RYEAR","RACCT","PRCTR","VERSN"],
        "keys": ["RCLNT","RLDNR","RBUKRS","RYEAR","RACCT","PRCTR","VERSN"],
        "types": {"RYEAR":"int", **{f"TSL{i:02d}":"decimal(23,2)" for i in range(1,17)}},
        "period_columns": [], "allowed": {}
    },
    "budget": {
        "columns": ["COMPANY_CODE","GL_ACCOUNT","COST_CENTER","PROFIT_CENTER","FISCAL_YEAR","FISCAL_PERIOD","BUDGET_AMOUNT_LOCAL","FORECAST_AMOUNT_LOCAL","PLAN_VERSION","DATA_CLASSIFICATION"],
        "required": ["COMPANY_CODE","GL_ACCOUNT","COST_CENTER","FISCAL_YEAR","FISCAL_PERIOD","BUDGET_AMOUNT_LOCAL"],
        "keys": ["COMPANY_CODE","GL_ACCOUNT","COST_CENTER","PROFIT_CENTER","FISCAL_YEAR","FISCAL_PERIOD","PLAN_VERSION"],
        "types": {"FISCAL_YEAR":"int","FISCAL_PERIOD":"int","BUDGET_AMOUNT_LOCAL":"decimal(23,2)","FORECAST_AMOUNT_LOCAL":"decimal(23,2)"},
        "period_columns": ["FISCAL_PERIOD"], "allowed": {}
    },
    "division_reference": {
        "columns": ["DIVISION_CODE", "DIVISION_NAME"],
        "required": ["DIVISION_CODE", "DIVISION_NAME"],
        "keys": ["DIVISION_CODE"], "types": {}, "period_columns": [], "allowed": {}
    },
    "gl_account_reference": {
        "columns": ["GL_ACCOUNT", "GL_ACCOUNT_NAME", "FINANCE_CATEGORY"],
        "required": ["GL_ACCOUNT", "GL_ACCOUNT_NAME", "FINANCE_CATEGORY"],
        "keys": ["GL_ACCOUNT"], "types": {}, "period_columns": [], "allowed": {}
    }
}

METADATA_COLUMNS = [
    "BATCH_RUN_ID", "EXTRACT_DATE", "LOAD_DATE", "SOURCE_SYSTEM",
    "SOURCE_FILE_NAME", "DQ_PASSED", "DQ_FAILURE_REASON"
]

## Reusable cleansing, casting and validation functions

In [ ]:
def table_exists(table_name):
    try:
        spark.table(table_name).limit(1).count()
        return True
    except Exception:
        return False


def standardize_column_names(df):
    seen = set()
    for old_name in df.columns:
        new_name = old_name.strip().upper().replace(" ", "_").replace("-", "_").replace(".", "_")
        if new_name in seen:
            raise ValueError(f"Duplicate column after standardisation: {new_name}")
        seen.add(new_name)
        if old_name != new_name:
            df = df.withColumnRenamed(old_name, new_name)
    return df


def cleanse_string_values(df):
    for field in df.schema.fields:
        if isinstance(field.dataType, T.StringType):
            cleaned = F.trim(F.regexp_replace(F.col(field.name), r"[\r\n\t]+", " "))
            df = df.withColumn(field.name, F.when(cleaned == "", None).otherwise(cleaned))
    return df


def parsed_value(column_name, target_type):
    raw = F.trim(F.col(column_name).cast("string"))
    if target_type == "date":
        return F.coalesce(
            F.to_date(raw, "yyyyMMdd"),
            F.to_date(raw, "yyyy-MM-dd"),
            F.to_date(raw, "dd.MM.yyyy")
        )
    if target_type.startswith("decimal"):
        normalised = F.regexp_replace(F.regexp_replace(raw, r"\s", ""), ",", "")
        return normalised.cast(target_type)
    return raw.cast(target_type)


def empty_reason_array():
    return F.array().cast("array<string>")


def append_reason(df, condition, reason):
    return df.withColumn(
        "_DQ_REASONS",
        F.when(condition, F.array_union(F.col("_DQ_REASONS"), F.array(F.lit(reason))))
         .otherwise(F.col("_DQ_REASONS"))
    )


def source_file_basename():
    return F.regexp_extract(F.input_file_name(), r"([^/?#]+)(?:[?#].*)?$", 1)


def successful_manifest_files(table_name):
    manifest_table = f"{AUD}.silver_file_manifest"
    if not table_exists(manifest_table):
        return None
    return (spark.table(manifest_table)
            .filter((F.col("TABLE_NAME") == table_name) & (F.col("STATUS") == "SUCCESS"))
            .select("SOURCE_FILE_NAME").distinct())

## Governed current-state upsert

Duplicate checks are scoped to `SOURCE_FILE_NAME + business key`. Valid rows are merged into Silver by business key, preventing full snapshots from creating duplicate history. A rerun safely reprocesses the same snapshot because the business-key MERGE is idempotent.


In [ ]:
def process_table(table_name, config):
    started = datetime.now(timezone.utc).replace(tzinfo=None)
    source_path = f"{BRONZE_ROOT}/{table_name}/{table_name}_{PROCESSING_DATE_KEY}.csv"

    raw = (spark.read
           .option("header", True)
           .option("inferSchema", False)
           .option("mode", "PERMISSIVE")
           .option("multiLine", False)
           .csv(source_path))

    raw = cleanse_string_values(standardize_column_names(raw))
    raw = (raw
           .withColumn("_SOURCE_FILE_PATH", F.input_file_name())
           .withColumn("SOURCE_FILE_NAME", source_file_basename()))

    manifest = successful_manifest_files(table_name) if SKIP_SUCCESSFULLY_PROCESSED_FILES else None
    if manifest is not None:
        raw = raw.join(F.broadcast(manifest), on="SOURCE_FILE_NAME", how="left_anti")

    discovered_files = [r["SOURCE_FILE_NAME"] for r in raw.select("SOURCE_FILE_NAME").distinct().collect()]
    if not discovered_files:
        return {
            "table_name": table_name, "status": "SKIPPED_NO_NEW_FILES", "source_rows": 0,
            "silver_rows": 0, "quarantine_rows": 0, "file_count": 0,
            "missing_columns": "", "unexpected_columns": "", "error": "",
            "started": started, "finished": datetime.now(timezone.utc).replace(tzinfo=None)
        }, [], []

    source_columns = [c for c in raw.columns if not c.startswith("_") and c != "SOURCE_FILE_NAME"]
    missing_columns = [c for c in config["required"] if c not in source_columns]
    unexpected_columns = [c for c in source_columns if c not in config["columns"]]

    # Add missing contract columns as null so every failed row can be quarantined with its lineage.
    for column_name in config["columns"]:
        if column_name not in raw.columns:
            raw = raw.withColumn(column_name, F.lit(None).cast("string"))

    projected_columns = list(config["columns"])
    if ALLOW_ADDITIVE_SOURCE_COLUMNS:
        projected_columns += [c for c in unexpected_columns if c not in projected_columns]
    df = raw.select(*projected_columns, "_SOURCE_FILE_PATH", "SOURCE_FILE_NAME")

    df = (df
          .withColumn("_EXTRACT_DATE_TEXT", F.regexp_extract(F.col("SOURCE_FILE_NAME"), r"(\d{8})", 1))
          .withColumn("EXTRACT_DATE", F.to_date(F.col("_EXTRACT_DATE_TEXT"), "yyyyMMdd"))
          .withColumn("LOAD_DATE", F.current_date())
          .withColumn("BATCH_RUN_ID", F.lit(BATCH_RUN_ID))
          .withColumn("SOURCE_SYSTEM", F.lit(SOURCE_SYSTEM))
          .withColumn("_DQ_REASONS", empty_reason_array()))

    # Filename date is lineage metadata. A malformed name creates a warning, not a row-level quarantine failure.
    df = (df
          .withColumn("DQ_WARNING",
                      F.when(F.col("EXTRACT_DATE").isNull(),
                             F.lit("INVALID_OR_MISSING_EXTRACT_DATE_IN_FILENAME")))
          .withColumn("EXTRACT_DATE",
                      F.coalesce(F.col("EXTRACT_DATE"), F.current_date())))
    if missing_columns:
        df = append_reason(df, F.lit(True), "SCHEMA_MISSING_REQUIRED_COLUMNS:" + ",".join(missing_columns))

    # Explicit type conversion with column-level failure reasons.
    for column_name, target_type in config.get("types", {}).items():
        original = F.trim(F.col(column_name).cast("string"))
        converted = parsed_value(column_name, target_type)
        cast_failed = original.isNotNull() & converted.isNull()
        df = append_reason(df, cast_failed, f"TYPE_CAST_FAILURE:{column_name}:{target_type}")
        df = df.withColumn(column_name, converted)

    # Required-field completeness after cleansing and casting.
    for column_name in config["required"]:
        null_condition = F.col(column_name).isNull() | (F.trim(F.col(column_name).cast("string")) == "")
        df = append_reason(df, null_condition, f"REQUIRED_FIELD_NULL:{column_name}")

    # Year and fiscal-period validity.
    for year_column in [c for c in ["GJAHR", "RYEAR", "FISCAL_YEAR"] if c in df.columns]:
        df = append_reason(df, F.col(year_column).isNotNull() & ~F.col(year_column).between(1900, 2100), f"INVALID_YEAR_RANGE:{year_column}")
    for period_column in config.get("period_columns", []):
        df = append_reason(df, F.col(period_column).isNotNull() & ~F.col(period_column).between(1, 16), f"INVALID_FISCAL_PERIOD:{period_column}")

    # Allowed-value checks.
    for column_name, allowed_values in config.get("allowed", {}).items():
        df = append_reason(df, F.col(column_name).isNotNull() & ~F.col(column_name).isin(allowed_values), f"INVALID_ALLOWED_VALUE:{column_name}")

    # Date interval consistency for master records.
    if "DATAB" in df.columns and "DATBI" in df.columns:
        df = append_reason(df, F.col("DATAB").isNotNull() & F.col("DATBI").isNotNull() & (F.col("DATAB") > F.col("DATBI")), "INVALID_DATE_INTERVAL:DATAB_GT_DATBI")

    # Duplicate business key within the same landed file. All duplicate members are quarantined.
    keys = [c for c in config["keys"] if c in df.columns]
    duplicate_window = Window.partitionBy("SOURCE_FILE_NAME", *keys)
    df = df.withColumn("_DUPLICATE_KEY_COUNT", F.count(F.lit(1)).over(duplicate_window))
    df = append_reason(df, F.col("_DUPLICATE_KEY_COUNT") > 1, "DUPLICATE_BUSINESS_KEY")

    df = (df
          .withColumn("DQ_PASSED", F.size(F.col("_DQ_REASONS")) == 0)
          .withColumn("DQ_FAILURE_REASON", F.when(F.col("DQ_PASSED"), F.lit(None).cast("string")).otherwise(F.concat_ws(";", F.col("_DQ_REASONS")))))

    cleanup_columns = ["_SOURCE_FILE_PATH", "_EXTRACT_DATE_TEXT", "_DQ_REASONS", "_DUPLICATE_KEY_COUNT"]
    valid = df.filter(F.col("DQ_PASSED")).drop(*cleanup_columns)
    rejected = df.filter(~F.col("DQ_PASSED")).drop(*cleanup_columns)

    source_rows = df.count()
    valid_rows = valid.count()
    rejected_rows = rejected.count()

    # Runtime DQ diagnostics before persistence.
    print(f"=== {table_name} DQ summary ===")
    df.groupBy("DQ_PASSED").count().show(truncate=False)
    if rejected_rows > 0:
        (rejected.groupBy("DQ_FAILURE_REASON").count()
         .orderBy(F.desc("count")).show(100, truncate=False))

    # Silver stores current state: update existing business keys and insert new keys.
    target = f"{SLV}.{table_name}"
    if valid_rows > 0:
        if table_exists(target):
            duplicate_target_keys = (spark.table(target)
                                     .groupBy(*keys).count()
                                     .filter(F.col("count") > 1)
                                     .limit(1).count())
            if duplicate_target_keys:
                raise RuntimeError(
                    f"{target} contains duplicate business keys from the former append-only design; "
                    "deduplicate it before enabling snapshot upserts"
                )
            merge_condition = " AND ".join(
                f"tgt.`{key}` <=> src.`{key}`" for key in keys
            )
            (DeltaTable.forName(spark, target).alias("tgt")
             .merge(valid.alias("src"), merge_condition)
             .whenMatchedUpdateAll()
             .whenNotMatchedInsertAll()
             .execute())
        else:
            (valid.write.format("delta").mode("overwrite")
             .option("mergeSchema", "false")
             .saveAsTable(target))

    if rejected_rows > 0:
        (rejected.write.format("delta").mode("append")
         .option("mergeSchema", "true")
         .saveAsTable(f"{QTN}.{table_name}"))

    # Rule-level counts. A row may contribute to multiple rule counts.
    rule_rows = []
    if rejected_rows > 0:
        exploded = (rejected
                    .select("SOURCE_FILE_NAME", F.explode(F.split("DQ_FAILURE_REASON", ";")).alias("DQ_RULE"))
                    .groupBy("SOURCE_FILE_NAME", "DQ_RULE").count())
        rule_rows = [(table_name, r["SOURCE_FILE_NAME"], r["DQ_RULE"], int(r["count"])) for r in exploded.collect()]

    file_rows = []
    for file_name in discovered_files:
        file_df = df.filter(F.col("SOURCE_FILE_NAME") == file_name)
        total = file_df.count()
        passed = file_df.filter(F.col("DQ_PASSED")).count()
        failed = total - passed
        status = "SUCCESS" if failed == 0 else ("PARTIAL_SUCCESS" if passed > 0 else "QUARANTINED")
        file_rows.append((table_name, file_name, total, passed, failed, status))

    finished = datetime.now(timezone.utc).replace(tzinfo=None)
    status = "SUCCESS" if rejected_rows == 0 else ("PARTIAL_SUCCESS" if valid_rows > 0 else "QUARANTINED")
    return {
        "table_name": table_name, "status": status, "source_rows": source_rows,
        "silver_rows": valid_rows, "quarantine_rows": rejected_rows,
        "file_count": len(discovered_files), "missing_columns": ",".join(missing_columns),
        "unexpected_columns": ",".join(unexpected_columns), "error": "",
        "started": started, "finished": finished
    }, file_rows, rule_rows

## Execute all table loads and persist operational audit tables

In [ ]:
run_results = []
manifest_results = []
rule_results = []
table_errors = []

for table_name, config in TABLE_CONFIG.items():
    try:
        result, file_rows, rule_rows = process_table(table_name, config)
        run_results.append(result)
        manifest_results.extend(file_rows)
        rule_results.extend(rule_rows)
        print(f"{table_name}: {result['status']} | source={result['source_rows']} silver={result['silver_rows']} quarantine={result['quarantine_rows']}")
    except Exception as error:
        now = datetime.now(timezone.utc).replace(tzinfo=None)
        message = str(error)[:2000]
        run_results.append({
            "table_name": table_name, "status": "ERROR", "source_rows": 0,
            "silver_rows": 0, "quarantine_rows": 0, "file_count": 0,
            "missing_columns": "", "unexpected_columns": "", "error": message,
            "started": now, "finished": now
        })
        table_errors.append((table_name, message))
        print(f"{table_name}: ERROR | {message}")

run_log_rows = [(
    BATCH_RUN_ID, r["table_name"], r["status"], r["source_rows"], r["silver_rows"],
    r["quarantine_rows"], r["file_count"], r["missing_columns"],
    r["unexpected_columns"], r["error"], r["started"], r["finished"]
) for r in run_results]

run_log = spark.createDataFrame(run_log_rows, """
    BATCH_RUN_ID string, TABLE_NAME string, STATUS string,
    SOURCE_ROWS long, SILVER_ROWS long, QUARANTINE_ROWS long, FILE_COUNT long,
    MISSING_COLUMNS string, UNEXPECTED_COLUMNS string, ERROR_MESSAGE string,
    STARTED_UTC timestamp, FINISHED_UTC timestamp
""")
run_log.write.format("delta").mode("append").saveAsTable(f"{AUD}.silver_load_log")

if manifest_results:
    manifest = spark.createDataFrame(manifest_results, """
        TABLE_NAME string, SOURCE_FILE_NAME string, SOURCE_ROWS long,
        SILVER_ROWS long, QUARANTINE_ROWS long, STATUS string
    """).withColumn("BATCH_RUN_ID", F.lit(BATCH_RUN_ID))        .withColumn("PROCESSED_UTC", F.current_timestamp())
    manifest.write.format("delta").mode("append").saveAsTable(f"{AUD}.silver_file_manifest")

if rule_results:
    rule_summary = spark.createDataFrame(rule_results, """
        TABLE_NAME string, SOURCE_FILE_NAME string, DQ_RULE string, FAILURE_COUNT long
    """).withColumn("BATCH_RUN_ID", F.lit(BATCH_RUN_ID))        .withColumn("PROCESSED_UTC", F.current_timestamp())
    rule_summary.write.format("delta").mode("append").saveAsTable(f"{AUD}.silver_dq_rule_summary")

display(run_log.orderBy("TABLE_NAME"))

non_successful = [
    (result["table_name"], result["status"], result["quarantine_rows"])
    for result in run_results
    if result["status"] != "SUCCESS"
]
if FAIL_NOTEBOOK_ON_TABLE_ERROR and (table_errors or non_successful):
    raise RuntimeError(
        "Silver snapshot was not fully successful; Bronze is retained for replay: "
        + str(non_successful or table_errors)
    )

## Reconciliation and integrity checks

For each processed snapshot, `SOURCE_ROWS = UPSERT_ROWS + QUARANTINE_ROWS`. Silver must contain one row per business key with valid lineage metadata and `DQ_PASSED = true`.


In [ ]:
reconciliation = (run_log
    .withColumn("RECONCILED", F.col("SOURCE_ROWS") == (F.col("SILVER_ROWS") + F.col("QUARANTINE_ROWS")))
    .select("TABLE_NAME", "STATUS", "SOURCE_ROWS", "SILVER_ROWS", "QUARANTINE_ROWS", "RECONCILED", "ERROR_MESSAGE"))
display(reconciliation.orderBy("TABLE_NAME"))

validation_rows = []
for table_name in TABLE_CONFIG:
    target = f"{SLV}.{table_name}"
    if table_exists(target):
        silver_df = spark.table(target)
        invalid_dq_flag = silver_df.filter(F.col("DQ_PASSED") != True).count()
        missing_lineage = silver_df.filter(
            F.col("BATCH_RUN_ID").isNull() |
            F.col("EXTRACT_DATE").isNull() |
            F.col("LOAD_DATE").isNull() |
            F.col("SOURCE_SYSTEM").isNull() |
            F.col("SOURCE_FILE_NAME").isNull()
        ).count()
        validation_rows.append((table_name, silver_df.count(), invalid_dq_flag, missing_lineage))

if validation_rows:
    validation = spark.createDataFrame(validation_rows, """
        TABLE_NAME string, SILVER_TOTAL_ROWS long,
        NON_PASSING_ROWS_IN_SILVER long, MISSING_LINEAGE_ROWS long
    """)
    display(validation.orderBy("TABLE_NAME"))

## Output contract

Valid Silver rows retain:

```text
BATCH_RUN_ID
EXTRACT_DATE
LOAD_DATE
SOURCE_SYSTEM
SOURCE_FILE_NAME
DQ_PASSED
DQ_FAILURE_REASON
```

Rejected rows retain the same metadata and their semicolon-delimited failure reasons in `quarantine.<table>`. Operational results are appended to:

```text
audit.silver_load_log
audit.silver_file_manifest
audit.silver_dq_rule_summary
```